In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)

N_NORMAL = 2000
N_FRAUD  = 100

normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, N_NORMAL).clip(5, 5000),
    'hour': np.random.normal(14, 4, N_NORMAL).clip(0, 23).astype(int),
    'is_electronics': np.random.binomial(1, 0.3, N_NORMAL),
    'tx_per_day': np.random.poisson(3, N_NORMAL),
    'fraud': 0
})

fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, N_FRAUD),
    'hour': np.random.choice([0,1,2,3,4,5,22,23], N_FRAUD),
    'is_electronics': np.random.binomial(1, 0.7, N_FRAUD),
    'tx_per_day': np.random.poisson(8, N_FRAUD),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

Dataset: 2100 wierszy, fraud rate: 4.8%


In [2]:
features = ['amount', 'hour', 'is_electronics', 'tx_per_day']
X = df[features]
y = df['fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

with open('fraud_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model zapisany do fraud_model.pkl")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00        20

    accuracy                           1.00       420
   macro avg       1.00      1.00      1.00       420
weighted avg       1.00      1.00      1.00       420

Model zapisany do fraud_model.pkl


In [3]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle, numpy as np

app = FastAPI(title="Fraud Detection API")
model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    hour: int
    is_electronics: int
    tx_per_day: int

@app.post("/score")
def score(tx: Transaction):
    features = np.array([[tx.amount, tx.hour, tx.is_electronics, tx.tx_per_day]])
    prediction = model.predict(features)[0]
    probability = model.predict_proba(features)[0][1]
    return {"is_fraud": bool(prediction), "fraud_probability": round(float(probability), 4)}

@app.get("/health")
def health():
    return {"status": "ok"}

Writing fraud_api.py


In [4]:
import requests

# Test normalna transakcja
r = requests.post("http://localhost:8001/score",
    json={"amount": 150, "hour": 14, "is_electronics": 0, "tx_per_day": 3})
print("Normalna:", r.json())

# Test podejrzana transakcja
r = requests.post("http://localhost:8001/score",
    json={"amount": 5500, "hour": 3, "is_electronics": 1, "tx_per_day": 12})
print("Podejrzana:", r.json())

# Test health
r = requests.get("http://localhost:8001/health")
print("Health:", r.json())

Normalna: {'is_fraud': False, 'fraud_probability': 0.0}
Podejrzana: {'is_fraud': True, 'fraud_probability': 1.0}
Health: {'status': 'ok'}


In [5]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json, requests

consumer = KafkaConsumer('transactions', bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', group_id='ml-scoring',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

API_URL = "http://localhost:8001/score"

for message in consumer:
    tx = message.value
    hour = tx.get('hour', int(tx['timestamp'][11:13]))
    is_elec = 1 if tx.get('category') == 'elektronika' else tx.get('is_electronics', 0)

    features = {
        "amount": tx['amount'],
        "hour": hour,
        "is_electronics": is_elec,
        "tx_per_day": 5
    }

    try:
        r = requests.post(API_URL, json=features)
        result = r.json()

        if result['is_fraud']:
            tx['ml_score'] = result['fraud_probability']
            alert_producer.send('alerts', value=tx)
            print(f"🚨 FRAUD | prob={result['fraud_probability']:.2f} | {tx['tx_id']} | {tx['amount']:.2f} PLN")
        else:
            print(f"   OK    | prob={result['fraud_probability']:.2f} | {tx['tx_id']} | {tx['amount']:.2f} PLN")
    except Exception as e:
        print(f"Błąd API: {e}")

Writing ml_consumer.py
